In [ ]:
#!/usr/bin/env python3

"""
Distinct 3-calendar-year co-occurrence episodes for double/triple disturbances.

For each pixel and each unordered combination, scan annual 0/1 observations
chronologically. An episode completes in the earliest year when every required
type has occurred at least once since the previous completion AND all their
most recent occurrences fall within the current three-calendar-year window.

After completion, reset that combination's history: no occurrence is reused
for another episode of the same combination.

Simultaneous occurrences and every ordering are allowed. Different combinations
are independent, so a triple episode can also contribute to double episodes.

Only observed years 2000-2020 are used. Outputs never overwrite existing files.
"""

from pathlib import Path
from contextlib import ExitStack
from collections import Counter
import csv
import numpy as np
import rasterio


# =============================================================================
# 1. PATHS AND SETTINGS
# =============================================================================

ROOT = Path('/home/jovyan/data-store/forest-disturbance-stack-v3')

BINARY_DIR = ROOT / 'data/derived/annual_stacks/binary'

FOREST_PATH = ROOT / 'data/derived/resampled/forest_mask_30m_resampled.tif'

OUT_DIR = ROOT / 'data/derived/disturbance_sequence/frequency_3yr_episodes'

YEARS = tuple(range(2000, 2021))

THRESHOLDS = ('any', 'extreme')

TYPES = ('wf', 'bt', 'hd')

COMBOS = {
    'wf_bt': (0, 1),
    'wf_hd': (0, 2),
    'bt_hd': (1, 2),
    'wf_bt_hd': (0, 1, 2),
}

WINDOW_YEARS = 3

NODATA = 255

PIXEL_AREA_KM2 = 0.0009


# =============================================================================
# 2. INPUT FILE PATHS
# =============================================================================

def annual_path(threshold, year):

    return BINARY_DIR / f'annual_stack_{threshold}_{year}.tif'


# =============================================================================
# 3. COUNT DISTINCT THREE-YEAR EPISODES
# =============================================================================

def count_episodes(history, indices, window_years=WINDOW_YEARS):

    """
    history: Boolean array [year, type, row, col]

    Returns a uint8 array containing the number of distinct episodes.

    A combination completes once all required disturbance types have
    occurred since the previous completion, with their most recent
    occurrence years spanning no more than 2 years.

    After completion, the history resets for that combination.
    """

    n_years, _, height, width = history.shape

    last_seen = np.full(
        (len(indices), height, width),
        -1000,
        dtype=np.int16
    )

    episodes = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    for t in range(n_years):

        # Update the most recent occurrence of each disturbance type
        for k, band_index in enumerate(indices):

            last_seen[k][history[t, band_index]] = t

        # Have all required disturbance types occurred?
        complete = np.all(last_seen >= 0, axis=0)

        # Time between earliest and latest required occurrences
        spread = last_seen.max(axis=0) - last_seen.min(axis=0)

        # A qualifying episode spans fewer than 3 calendar-year steps
        new_episode = complete & (spread < window_years)

        # Count completed episodes
        episodes += new_episode.astype(np.uint8)

        # Reset after completion to avoid reusing occurrences
        last_seen[:, new_episode] = -1000

    return episodes


# =============================================================================
# 4. VALIDATION TESTS
# =============================================================================

def self_test():

    def run(events, combo):

        h = np.zeros((21, 3, 1, 1), dtype=bool)

        for year, types in events.items():

            for d in types:

                h[
                    year - 2000,
                    TYPES.index(d),
                    0,
                    0
                ] = True

        return int(
            count_episodes(h, COMBOS[combo])[0, 0]
        )

    tests = [

        # Simultaneous WF + BT followed by HD
        (
            {2001: ('wf', 'bt'), 2003: ('hd',)},
            'wf_bt_hd',
            1
        ),

        # HD -> BT -> WF
        (
            {2001: ('hd',), 2002: ('bt',), 2003: ('wf',)},
            'wf_bt_hd',
            1
        ),

        # BT -> HD
        (
            {2001: ('bt',), 2003: ('hd',)},
            'bt_hd',
            1
        ),

        # HD -> BT
        (
            {2001: ('hd',), 2003: ('bt',)},
            'bt_hd',
            1
        ),

        # Four calendar years: does not qualify
        (
            {2001: ('wf',), 2002: ('bt',), 2004: ('hd',)},
            'wf_bt_hd',
            0
        ),

        # All three disturbances in the same year
        (
            {2001: ('wf', 'bt', 'hd')},
            'wf_bt_hd',
            1
        ),

        # Repeated BT does not create an additional triple
        (
            {
                2001: ('wf', 'bt'),
                2002: ('bt',),
                2003: ('hd',)
            },
            'wf_bt_hd',
            1
        ),

        # Two separate triple episodes
        (
            {
                2001: ('wf', 'bt'),
                2003: ('hd',),
                2005: ('wf',),
                2006: ('bt',),
                2007: ('hd',)
            },
            'wf_bt_hd',
            2
        ),

        # Beginning of the observation period
        (
            {2000: ('wf',), 2002: ('bt', 'hd')},
            'wf_bt_hd',
            1
        ),

        # End of the observation period
        (
            {2018: ('wf',), 2020: ('bt', 'hd')},
            'wf_bt_hd',
            1
        ),

        # Repeated BT with only one WF
        (
            {
                2001: ('wf',),
                2002: ('bt',),
                2003: ('bt',)
            },
            'wf_bt',
            1
        ),

        # Two separate WF + BT episodes
        (
            {
                2001: ('wf',),
                2002: ('bt',),
                2003: ('wf',),
                2004: ('bt',)
            },
            'wf_bt',
            2
        ),
    ]

    for events, combo, expected in tests:

        got = run(events, combo)

        assert got == expected, (
            events, combo, got, expected
        )

    print(f'Passed {len(tests)} episode-counting tests.')


# =============================================================================
# 5. MAIN PROCESSING
# =============================================================================

def main():

    self_test()

    OUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # -------------------------------------------------------------------------
    # Define output rasters
    # -------------------------------------------------------------------------

    output_paths = {

        (threshold, combo):
            OUT_DIR / f'{combo}_frequency_3yr_{threshold}.tif'

        for threshold in THRESHOLDS
        for combo in COMBOS

    }

    summary_path = (
        OUT_DIR / 'frequency_3yr_episode_distribution.csv'
    )

    # Prevent accidental overwriting
    for path in [*output_paths.values(), summary_path]:

        if path.exists():

            raise FileExistsError(
                f'Refusing to overwrite existing file: {path}'
            )

    distributions = Counter()

    # -------------------------------------------------------------------------
    # Open input rasters
    # -------------------------------------------------------------------------

    with ExitStack() as stack:

        forest_src = stack.enter_context(
            rasterio.open(FOREST_PATH)
        )

        annual = {

            (threshold, year):
                stack.enter_context(
                    rasterio.open(annual_path(threshold, year))
                )

            for threshold in THRESHOLDS
            for year in YEARS

        }

        # ---------------------------------------------------------------------
        # Verify raster alignment
        # ---------------------------------------------------------------------

        for src in annual.values():

            if (
                src.width != forest_src.width
                or src.height != forest_src.height
                or src.transform != forest_src.transform
                or src.crs != forest_src.crs
            ):

                raise ValueError(
                    f'Grid mismatch: {src.name}'
                )

            if src.count < 3:

                raise ValueError(
                    f'Expected 3+ bands: {src.name}'
                )

        # ---------------------------------------------------------------------
        # Set output raster properties
        # ---------------------------------------------------------------------

        profile = forest_src.profile.copy()

        profile.update(
            driver='GTiff',
            count=1,
            dtype='uint8',
            nodata=NODATA,
            compress='deflate',
            predictor=2,
            tiled=True,
            BIGTIFF='IF_SAFER'
        )

        # Create eight output rasters
        destinations = {

            key:
                stack.enter_context(
                    rasterio.open(path, 'w', **profile)
                )

            for key, path in output_paths.items()

        }

        # ---------------------------------------------------------------------
        # Process raster blocks
        # ---------------------------------------------------------------------

        for block_no, (_, window) in enumerate(
            forest_src.block_windows(1), 1
        ):

            if block_no % 100 == 0:

                print(
                    f'Processed {block_no:,} blocks...',
                    flush=True
                )

            # Forest pixels only
            forest = (
                forest_src.read(1, window=window) == 1
            )

            # Skip blocks without forest
            if not forest.any():

                empty = np.full(
                    forest.shape,
                    NODATA,
                    dtype=np.uint8
                )

                for dst in destinations.values():

                    dst.write(
                        empty,
                        1,
                        window=window
                    )

                continue

            # -----------------------------------------------------------------
            # Process any and extreme thresholds separately
            # -----------------------------------------------------------------

            for threshold in THRESHOLDS:

                # Shape:
                # [21 years, 3 disturbance types, rows, columns]

                history = np.stack([

                    annual[threshold, year].read(
                        [1, 2, 3],
                        window=window
                    ) == 1

                    for year in YEARS

                ], axis=0)

                # -------------------------------------------------------------
                # Calculate each double and triple combination
                # -------------------------------------------------------------

                for combo, indices in COMBOS.items():

                    result = count_episodes(
                        history,
                        indices
                    )

                    # Set nonforest pixels to NoData
                    result[~forest] = NODATA

                    # Write output raster
                    destinations[
                        threshold, combo
                    ].write(
                        result,
                        1,
                        window=window
                    )

                    # ---------------------------------------------------------
                    # Summarize episode frequencies
                    # ---------------------------------------------------------

                    values, counts = np.unique(
                        result[forest],
                        return_counts=True
                    )

                    for value, n in zip(values, counts):

                        distributions[
                            threshold,
                            combo,
                            int(value)
                        ] += int(n)

    # =========================================================================
    # 6. WRITE SUMMARY CSV
    # =========================================================================

    with open(
        summary_path,
        'x',
        newline=''
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            'threshold',
            'combination',
            'episode_count',
            'pixel_count',
            'area_km2'
        ])

        for (threshold, combo, episodes), n in sorted(
            distributions.items()
        ):

            writer.writerow([
                threshold,
                combo,
                episodes,
                n,
                n * PIXEL_AREA_KM2
            ])

    # =========================================================================
    # 7. FINAL VALIDATION
    # =========================================================================

    for threshold in THRESHOLDS:

        totals = [

            sum(
                n
                for (t, c, _), n in distributions.items()
                if t == threshold and c == combo
            )

            for combo in COMBOS

        ]

        assert len(set(totals)) == 1, (
            f'Inconsistent forest pixel counts: {threshold}'
        )

    # =========================================================================
    # 8. FINISHED
    # =========================================================================

    print(
        'Completed. New rasters and summary CSV saved to:',
        OUT_DIR
    )

    for path in [*output_paths.values(), summary_path]:

        print(' ', path.name)


if __name__ == '__main__':
    main()

Passed 12 episode-counting tests.
Processed 100 blocks...
Processed 200 blocks...
Processed 300 blocks...
Processed 400 blocks...
Processed 500 blocks...
Processed 600 blocks...
Processed 700 blocks...
Processed 800 blocks...
Processed 900 blocks...
Processed 1,000 blocks...
Processed 1,100 blocks...
Processed 1,200 blocks...
Processed 1,300 blocks...
Processed 1,400 blocks...
Processed 1,500 blocks...
Processed 1,600 blocks...
Processed 1,700 blocks...
Processed 1,800 blocks...
Processed 1,900 blocks...
Processed 2,000 blocks...
Processed 2,100 blocks...
Processed 2,200 blocks...
Processed 2,300 blocks...
Processed 2,400 blocks...
Processed 2,500 blocks...
Processed 2,600 blocks...
Processed 2,700 blocks...
Processed 2,800 blocks...
Processed 2,900 blocks...
Processed 3,000 blocks...
Processed 3,100 blocks...
Processed 3,200 blocks...
Processed 3,300 blocks...
Processed 3,400 blocks...
Processed 3,500 blocks...
Processed 3,600 blocks...
Processed 3,700 blocks...
Processed 3,800 blocks